# CohereX on Google Colab

Transcribe audio with word-level timestamps and (optionally) speaker labels.

Set the runtime to a GPU first: **Runtime → Change runtime type → GPU**.

In [ ]:
!apt-get -qq install -y ffmpeg
# 2>/dev/null hides Colab's harmless pip dependency-resolver warnings.
!pip install -q coherex 2>/dev/null

Log in to Hugging Face (needs access to the gated [Cohere Transcribe](https://huggingface.co/CohereLabs/cohere-transcribe-03-2026) model).

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

Upload an audio file.

In [ ]:
from google.colab import files
audio_path = next(iter(files.upload()))

Transcribe. Change `--language` to your audio's language, or use `--diarize` for speaker labels.

In [ ]:
!coherex "{audio_path}" --language en --vad_method silero --max_line_width 42 --max_line_count 2 -o out/

For Arabic, English, or Arabic-English code-switched audio, the finetuned [`cohere-transcribe-arabic-07-2026`](https://huggingface.co/CohereLabs/cohere-transcribe-arabic-07-2026) model is more accurate.

In [ ]:
!coherex "{audio_path}" --model CohereLabs/cohere-transcribe-arabic-07-2026 --language ar --vad_method silero --max_line_width 42 --max_line_count 2 -o out-ar/

Add speaker labels with `--diarize`. This needs access to the gated [`speaker-diarization-community-1`](https://huggingface.co/pyannote/speaker-diarization-community-1) model (accept its terms first).

In [ ]:
from huggingface_hub import get_token
!coherex "{audio_path}" --language en --diarize --hf_token {get_token()} --vad_method silero --max_line_width 42 --max_line_count 2 -o out-diarize/

Outputs (SRT, VTT, TXT, TSV, JSON) are written to the `out/` folder in the file browser on the left.

---
## Optional: vLLM backend

Higher throughput for many/long files. It is **slower for a single short clip** (the server has to start and load the model first), so prefer the default backend above for quick jobs.

Run the install cell, then **Runtime → Restart session** (vLLM brings its own CUDA build), and finally run the last cell.

In [ ]:
# Installs vLLM with a matching CUDA build.
# After this finishes: Runtime -> Restart session, then run the next cell.
!pip install -q "coherex[vllm]"

In [ ]:
# Run after restarting the session.
# CohereX starts a vLLM server, transcribes, then stops it automatically.
# Cap vLLM's GPU memory so forced alignment still fits afterwards.
audio_path = "voice-sample-1.mp3"  # name of the file you uploaded above
!coherex "{audio_path}" --language en --backend vllm --vllm_args "--gpu-memory-utilization 0.6" -o out-vllm/